In [1]:
!pip install autogen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 994.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.0 MB/s eta 0:00:00


In [2]:
import autogen
import os
from getpass import getpass

os.environ["AUTOGEN_USE_DOCKER"] = "False"
OPENAI_API_KEY = getpass("Enter your API key here: ")

config_list = [{"model": "gpt-4o", "api_key": OPENAI_API_KEY}]   # no quotes
llm_config = {"config_list": config_list}

Enter your API key here: ··········


In [3]:
from autogen import AssistantAgent, UserProxyAgent

assistant=AssistantAgent("assistant", llm_config=llm_config)
user_proxy = UserProxyAgent("user_proxy", human_input_mode="NEVER",llm_config=llm_config)

user_proxy.initiate_chat(assistant, message="Hello!  How are you today?", max_turns=1)

user_proxy (to assistant):

Hello!  How are you today?

--------------------------------------------------------------------------------
assistant (to user_proxy):

Hello! I'm here and ready to assist you with any tasks or questions you might have. How can I help you today?

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (83b4c333-6214-4b0a-80d9-a8ada84eec44): Maximum turns (1) reached


ChatResult(chat_id=93420034015150988068515779044826762482, chat_history=[{'content': 'Hello!  How are you today?', 'role': 'assistant', 'name': 'user_proxy'}, {'content': "Hello! I'm here and ready to assist you with any tasks or questions you might have. How can I help you today?", 'role': 'user', 'name': 'assistant'}], summary="Hello! I'm here and ready to assist you with any tasks or questions you might have. How can I help you today?", cost={'usage_including_cached_inference': {'total_cost': 0.001435, 'gpt-4o-2024-08-06': {'cost': 0.001435, 'prompt_tokens': 474, 'completion_tokens': 25, 'total_tokens': 499}}, 'usage_excluding_cached_inference': {'total_cost': 0.001435, 'gpt-4o-2024-08-06': {'cost': 0.001435, 'prompt_tokens': 474, 'completion_tokens': 25, 'total_tokens': 499}}}, human_input=[])

In [4]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 796.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found ex

In [5]:
import chromadb

chroma_client = chromadb.Client()

products_collection = chroma_client.create_collection(name="products")
orders_collection = chroma_client.create_collection(name="orders")

products_collection.add(
    documents=["Frisbee", "Dog Bowl"],
    ids=["id1", "id2"],
)

orders_collection.add(documents=["1234", "5678"], ids=["id1", "id2"])


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 24.6MiB/s]


In [6]:
import csv

with open("/content/products.csv") as file:
    product_lines = csv.reader(file)

    id = 1

    for line in product_lines:
        if line[0] == "Product Name":
            continue
        else:
            products_collection.add(documents=str(line), ids=str(id))
            id += 1

with open("/content/orders.csv") as file:
    order_lines = csv.reader(file)

    id = 1

    for line in order_lines:
        if line[0] == "Order Number":
            continue
        else:
            orders_collection.add(documents=str(line), ids=str(id))
            id += 1



In [7]:
def search_products(search_query):
    results = products_collection.query(query_texts=[search_query], n_results=2)
    return results


def search_orders(search_query):
    results = orders_collection.query(query_texts=[search_query], n_results=2)
    return results


products_search_results = search_products("What is Artisanal Air?")
print(products_search_results["documents"])

orders_search_results = search_orders("Earl Eclair")
print(orders_search_results["documents"])

[["['Artisanal Air', '100% genuine Brooklyn oxygen (may contain subway fumes). Sold in vintage mason jar.', '39.99', '1.1']", "['Fake Plant That Collects Real Dust', 'Thrives in neglect. The only thing growing is your guilt for not cleaning.', '19.99', '2.5']"]]
[["['ORD1013', 'Baron Von Clownface', '25.98', 'Disappointment Alarm Clock, Anti-Social Social Media Gloves']", "['ORD1028', 'Earl Eclair', '28.97', 'Unnecessarily Complex Can Opener, Single-Use Reusable Bag, Ethical Socks (One Missing)']"]]


In [8]:
import asyncio
from typing_extensions import Annotated

# config_list comes from cell 1 (getpass) - do NOT redefine it here

products_search_assistant_agent = autogen.AssistantAgent(
        name="products_search_assistant",
        system_message="""You are a helpful assistant.
You have access to a database of documents about products, and you may search it.
The correct syntax for a search is: "what you want to search for".
Please use the search function to find information which can be used to answer the user's question.
DO NOT rely on your own knowledge, ONLY use the information retrieved from the search.
Please be honest about what you find. I am not looking for perfection, just the truth.
You are amazing and you can do this. I will pay you $200 for an excellent result,
but only if you follow all instructions exactly.""",
        llm_config={
            "config_list": config_list,
            "temperature": 0.0
        },
        code_execution_config=False,
    )

orders_search_assistant_agent = autogen.AssistantAgent(
        name="orders_search_assistant",
        system_message="""You are a helpful assistant.
You have access to a database of documents about customer orders,
and you may search it. The correct syntax for a search is: "what you want to search for".
Please use the search function to find information which can be used to answer the user's question.
DO NOT rely on your own knowledge, ONLY use the information retrieved from the search.
Please be honest about what you find. I am not looking for perfection, just the truth.
You are amazing and you can do this. I will pay you $200 for an excellent result,
but only if you follow all instructions exactly.""",
        llm_config={
            "config_list": config_list,
            "temperature": 0.0
        },
        code_execution_config=False,
    )


In [9]:
products_search_executor_agent = autogen.UserProxyAgent(
        name="products_search_executor",
        code_execution_config=False,
        system_message="""When enough information has been retrieved to answer
          the user's question to full satisfaction,
          please return "TERMINATE" to end the conversation.
        If more information must be collected, please return CONTINUE.""",
        human_input_mode="NEVER",
        max_consecutive_auto_reply=4,
        is_termination_msg=lambda x: x.get("content", "")
        .rstrip()
        .endswith("TERMINATE"),
        llm_config={
            "config_list": config_list,
            "temperature": 0.0
        },
    )

orders_search_executor_agent = autogen.UserProxyAgent(
        name="orders_search_executor",
        code_execution_config=False,
        system_message="""When enough information has been retrieved to answer
          the user's question to full satisfaction,
          please return "TERMINATE" to end the conversation.
        If more information must be collected, please return CONTINUE.""",
        human_input_mode="NEVER",
        max_consecutive_auto_reply=4,
        is_termination_msg=lambda x: x.get("content", "")
        .rstrip()
        .endswith("TERMINATE"),
        llm_config={
            "config_list": config_list,
            "temperature": 0.0
        },
    )


In [10]:
products_groupchat = autogen.GroupChat(
        agents=[products_search_assistant_agent,
                products_search_executor_agent],
        messages=[],
        max_round=8,
        speaker_selection_method="round_robin",
    )

products_groupchat_manager = autogen.GroupChatManager(
        groupchat=products_groupchat,
        name="products_groupchat_manager",
        llm_config={"config_list": config_list},
        is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    )

orders_groupchat = autogen.GroupChat(
        agents=[orders_search_assistant_agent,
                orders_search_executor_agent],
        messages=[],
        max_round=8,
        speaker_selection_method="round_robin",
    )

orders_groupchat_manager = autogen.GroupChatManager(
        groupchat=orders_groupchat,
        name="orders_groupchat_manager",
        llm_config={"config_list": config_list},
        is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    )


In [11]:
async_chat_plan = [
        {
            "chat_id": 1,
            "recipient": products_groupchat_manager,
            "message": "What is Artisanal Air?.",
            "summary_method": "reflection_with_llm",
            "silent": False,
        },
        {
            "chat_id": 2,
            "recipient": orders_groupchat_manager,
            "message": "What did Ned Noodle order?",
            "summary_method": "reflection_with_llm",
            "silent": False,

        }
    ]


async def start_groupchat():
    user = autogen.UserProxyAgent(
        name="User",
        human_input_mode="NEVER",
        is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    )
    await user.a_initiate_chats(async_chat_plan)

# Run this only in cell 12, AFTER the tools are registered (cell 11)

In [12]:
@products_search_executor_agent.register_for_execution()
@products_search_assistant_agent.register_for_llm(description="Search a ChromaDB collection containing information about products.")
async def search_products(search_query: Annotated[str, "Search query"]):
    results = products_collection.query(query_texts=[search_query], n_results=2)
    return results["documents"]

@orders_search_executor_agent.register_for_execution()
@orders_search_assistant_agent.register_for_llm(description="Search a ChromaDB collection containing information about customer orders.")
async def search_orders(search_query: Annotated[str, "Search query"]):
    results = orders_collection.query(query_texts=[search_query], n_results=2)
    return results["documents"]


In [13]:
await start_groupchat()
#asyncio.run(start_groupchat())


********************************************************************************
Starting a new chat....

********************************************************************************

********************************************************************************
Starting a new chat....

********************************************************************************
User (to products_groupchat_manager):

What is Artisanal Air?.

--------------------------------------------------------------------------------

Next speaker: products_search_assistant

User (to orders_groupchat_manager):

What did Ned Noodle order?

--------------------------------------------------------------------------------

Next speaker: orders_search_assistant

orders_search_assistant (to orders_groupchat_manager):

***** Suggested tool call (call_uWSVgbdOTNjtZAiARkCrAAfy): search_orders *****
Arguments: 
{"search_query":"Ned Noodle order"}
********************************************************************

In [14]:
writer_assistant_agent = autogen.AssistantAgent(
        name="writer_assistant",
        system_message="""You are a helpful assistant for a company.
Your job is to answer the user's question using the provided information.
DO NOT rely on your own knowledge, ONLY use the provided info.
If you don't know the answer, just say you don't know.
You are amazing and you can do this. I will pay you $200 for an excellent result, but only if you follow all instructions exactly.""",
        llm_config={
            "config_list": config_list,
            "temperature": 0,
        },
        code_execution_config=False,
        max_consecutive_auto_reply=1,
    )


In [15]:
retrieved_product_data = products_search_assistant_agent.chat_messages
retrieved_order_data = orders_search_assistant_agent.chat_messages

retrieved_data = "Retrieved product data: " + str(retrieved_product_data) + "  Retrieved order data: " + str(retrieved_order_data)
print(retrieved_data)

Retrieved product data: defaultdict(<class 'list'>, {<autogen.agentchat.groupchat.GroupChatManager object at 0x7b455cd35010>: [{'content': 'What is Artisanal Air?.', 'name': 'User', 'role': 'user'}, {'tool_calls': [{'id': 'call_mWCzhwe1Oz2fOq1WuUMoeouC', 'function': {'arguments': '{"search_query":"Artisanal Air"}', 'name': 'search_products'}, 'type': 'function'}], 'content': None, 'role': 'assistant'}, {'content': '[["[\'Artisanal Air\', \'100% genuine Brooklyn oxygen (may contain subway fumes). Sold in vintage mason jar.\', \'39.99\', \'1.1\']", "[\'Reverse Umbrella\', \'Innovative design soaks your pants first. Now with 360-degree drippage technology!\', \'29.99\', \'1.7\']"]]', 'tool_responses': [{'tool_call_id': 'call_mWCzhwe1Oz2fOq1WuUMoeouC', 'role': 'tool', 'content': '[["[\'Artisanal Air\', \'100% genuine Brooklyn oxygen (may contain subway fumes). Sold in vintage mason jar.\', \'39.99\', \'1.1\']", "[\'Reverse Umbrella\', \'Innovative design soaks your pants first. Now with 36

In [16]:
user_question = "What is Artisanal Air?"

writer_prompt = f"""Please write the final answer to the user's question: \n{user_question}\n\n
     The information retrieved from the search agents is:
     {retrieved_data}.  I will tip you $200 for an excellent result."""


writer_userproxy = autogen.UserProxyAgent(
        name="WriterUserproxy",  # no spaces in the agent name
        human_input_mode="NEVER",
        is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    )

writer_userproxy.initiate_chat(
        recipient=writer_assistant_agent, message=writer_prompt, silent=True
    )

result = writer_assistant_agent.chat_messages
print("FINAL RESULT:")
cleaned_result = list(result.items())[0][1][-2]["content"]
print(cleaned_result)


>>>>>>>> TERMINATING RUN (6673a66c-7850-4166-a191-7e6b327e6662): Maximum number of consecutive auto-replies reached
FINAL RESULT:
Artisanal Air is described as "100% genuine Brooklyn oxygen (may contain subway fumes)" and is sold in a vintage mason jar. It is priced at $39.99.


In [17]:
#More questions to ask:

#What is Stain Magnet T-Shirt?
#What is Disappointment Alarm Clock?
#Fake Plant That Collects Real Dust

#What did Admiral Bananaface order?
#Did Sergeant Noodles order a non-stick pan?
#What did Waffles McSprinkles order?

In [18]:
# while True:
#   answer = rag(user_question)
#   if answer is True:
#     break
#   else:
#     fix(function) OR fix (prompt)